In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import v2

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [15]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3,6,5),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(6,16,5),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Flatten(),
            nn.Linear(16*5*5,120),
            nn.ReLU(),
            nn.Linear(120,60),
            nn.ReLU(),
            nn.Linear(60,10)
         )

    def forward(self,x):
        return self.layers(x)


In [8]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

learning_rate = 0.01

epochs = 10

trainset = datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

/home/redinent/coding/trash/env/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [16]:
model = CNN()

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [19]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(correct*100)

In [18]:
for i in range(epochs):
    train_loop(trainloader,model,loss_fn,optimizer)
    print(epochs)

test_loop(testloader,model,loss_fn)


loss: 2.346919  [    8/50000]
loss: 2.299097  [  808/50000]
loss: 2.320153  [ 1608/50000]
loss: 2.288316  [ 2408/50000]
loss: 2.282938  [ 3208/50000]
loss: 2.280133  [ 4008/50000]
loss: 2.285984  [ 4808/50000]
loss: 2.325457  [ 5608/50000]
loss: 2.276725  [ 6408/50000]
loss: 2.273969  [ 7208/50000]
loss: 2.287494  [ 8008/50000]
loss: 2.321991  [ 8808/50000]
loss: 2.252305  [ 9608/50000]
loss: 1.998186  [10408/50000]
loss: 2.213598  [11208/50000]
loss: 1.942294  [12008/50000]
loss: 1.847070  [12808/50000]
loss: 2.482756  [13608/50000]
loss: 2.040182  [14408/50000]
loss: 2.176531  [15208/50000]
loss: 1.539103  [16008/50000]
loss: 1.920616  [16808/50000]
loss: 1.868779  [17608/50000]
loss: 1.650014  [18408/50000]
loss: 2.207690  [19208/50000]
loss: 1.875636  [20008/50000]
loss: 1.828970  [20808/50000]
loss: 1.603573  [21608/50000]
loss: 1.525358  [22408/50000]
loss: 1.610991  [23208/50000]
loss: 1.492067  [24008/50000]
loss: 2.016132  [24808/50000]
loss: 1.891187  [25608/50000]
loss: 0.99

In [21]:
test_loop(testloader,model,loss_fn)

63.77
